# 一、CVAE

In [15]:
# 准备环境等
import torch, torchvision
import torch.nn as nn
import torch.nn.functional as F

dataset = torchvision.datasets.MNIST(root='./.data', transform=torchvision.transforms.ToTensor(),)


class Encoder(nn.Module):
    def __init__(self, hidden_size, embed_size, n_fc=1):
        super().__init__()
        hidden_size = hidden_size//2 * 2
        self.img_size = 28 * 28
        self.n_num = 10

        self.fc_img = nn.Linear(self.img_size, hidden_size//2)
        self.fc_onehot = nn.Linear(self.n_num, hidden_size//2)
        self.n_fc = n_fc

        self.fcs_log_var = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(self.n_fc)])
        self.fcs_mean = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(self.n_fc)])

        self.fc_out_log_var = nn.Linear(hidden_size, embed_size)
        self.fc_out_mean = nn.Linear(hidden_size, embed_size)

    def forward(self, img, label):
        img = self.fc_img(img) # (*, 784) -> (*, hidden_size//2)
        label = F.one_hot(label, self.n_num).to(torch.float)  # (*) -> (*, hidden_size//2)
        label = self.fc_onehot(label)
        x = torch.cat((img, label), dim=-1)
        log_var = x
        mean = x

        for fc in self.fcs_log_var:
            log_var = F.relu(fc(log_var))

        log_var = self.fc_out_log_var(log_var)
        for fc in self.fcs_mean:
            mean = F.relu(fc(mean))
        mean = self.fc_out_mean(mean)
        return mean, log_var


class Decoder(nn.Module):
    def __init__(self, hidden_size, embed_size, n_fc=1):
        super().__init__()
        self.embed_size = embed_size
        self.hidden_size = hidden_size = hidden_size//2 * 2
        self.img_size = 28 * 28
        self.n_num = 10
        self.n_fc = n_fc

        self.fc_noise = nn.Linear(self.embed_size, hidden_size//2)
        self.fc_label = nn.Linear(self.n_num, hidden_size//2)

        self.fcs = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(n_fc)])
        self.fc_out = nn.Linear(hidden_size, self.img_size)

    def forward(self, noise, label):
        # noise : (*, embed_size)
        # label : (*)
        noise = self.fc_noise(noise)
        label = self.fc_label(F.one_hot(label, self.n_num).to(torch.float))
        x = torch.cat((noise, label), dim=-1)
        for fc in self.fcs:
            x = F.relu(fc(x))
        x = self.fc_out(x)
        return torch.sigmoid(x)

i, l = dataset[0]
i = i.flatten().unsqueeze(0)
l = torch.tensor(l).unsqueeze(0)

HIDDEN_SIZE = 512
EMBED_SIZE = 32
N_FC = 1

# # 测试一下Encoder和Decoder
# decoder = Decoder(HIDDEN_SIZE, EMBED_SIZE, n_fc=N_FC)
# encoder = Encoder(HIDDEN_SIZE, EMBED_SIZE, n_fc=N_FC)
#
# m, log_v = encoder(i, l)
# im = decoder(m, l)


In [ ]:
# 搭建训练框架
LR = 1e-4
epoch = 40
